In [ ]:
import sys
import os
import time
import torch

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [ ]:
class Buffer:
    def __init__(self, buffer_capacity=530000, num_states=14):
        # Number of "experiences" to store at max
        self.buffer_capacity = buffer_capacity


        # Its tells us num of times record() was called.
        self.buffer_counter = 0

        # Instead of list of tuples as the exp.replay concept go
        # We use different np.arrays for each tuple element
        self.state_buffer = np.zeros((self.buffer_capacity, num_states))
        self.action_buffer = np.zeros((self.buffer_capacity, 1))
        self.reward_buffer = np.zeros((self.buffer_capacity, 1))
        self.done_buffer = np.zeros((self.buffer_capacity, 1))
        self.next_state_buffer = np.zeros((self.buffer_capacity, num_states))

    # Takes (s,a,r,s') obervation tuple as input
    def record(self, obs_tuple):
        # Set index to zero if buffer_capacity is exceeded,
        # replacing old records
        index = self.buffer_counter % self.buffer_capacity

        self.state_buffer[index] = obs_tuple[0]
        self.action_buffer[index] = obs_tuple[1]
        self.reward_buffer[index] = obs_tuple[2]
        self.next_state_buffer[index] = obs_tuple[3]
        self.done_buffer[index] = obs_tuple[4]

        self.buffer_counter += 1

In [ ]:
save_dir = './final_result/1hourly/'

full_data_file = os.path.join(save_dir, 'heparin_final_data_withTimes.csv')
full_zs = pd.read_csv(full_data_file)

In [ ]:
## Determine the train, val, test split (70/15/15), stratified by patient outcome
print(len(full_zs))
all_id = full_zs.groupby('traj')['PTT_reward'].count().index.values
print(len(all_id))
print("----------------")

temp = full_zs.loc[full_zs['PTT_reward']>=0,:]
print(len(temp))
temp = temp.groupby('traj')['PTT_reward'].count()
print(len(temp))

보상값을 기준으로 train_test_split을 하려고 하는데 이 때 기준을 보상값이 >=0 이상인 행들을 trajectory마다 개수를 카운트 한 다음에 이 값의 중위값을 기준으로 pos_cohort, neg_cohort로 나눔.

In [ ]:
pos_cohort = list()
neg_cohort = list()
for j,i in zip(temp.index.values, temp.values):
    if i >=np.median(temp.values):
        pos_cohort.append(j)
    else:
        neg_cohort.append(j)

print(len(pos_cohort), len(neg_cohort))
        

In [ ]:
pos_neg_dict = {}

pos_neg = pos_cohort + neg_cohort
much_neg_cohort = [i for i in all_id if i not in pos_neg]

for i in pos_cohort:
    pos_neg_dict[i]=1
for i in neg_cohort:
    pos_neg_dict[i]=-1
for i in much_neg_cohort:
    pos_neg_dict[i]=-1


len(pos_neg_dict)

In [ ]:
pos_neg_dict = sorted(pos_neg_dict.items())

X = [i[0] for i in pos_neg_dict]
y = [i[1] for i in pos_neg_dict]

print(len(X), len(y))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, stratify=y_test, test_size=0.5)

In [ ]:
full_zs = full_zs.drop(['m:presumed_onset', 'm:charttime', 'm:icustayid'], axis=1)

In [ ]:
train_data = full_zs[full_zs['traj'].isin(X_train)]
trajectories = train_data['traj'].unique()

val_data = full_zs[full_zs['traj'].isin(X_val)]
val_trajectories = val_data['traj'].unique()

test_data = full_zs[full_zs['traj'].isin(X_test)]
test_trajectories = test_data['traj'].unique()

In [ ]:
# Define the features of the full data
num_actions = 1
# state_dim = 14
num_obs = 14
num_dem = 5
# num_acuity_scores = 3
# horizon = 21
device = 'cpu'



# train buffer 저장

In [ ]:
dem_keep_cols = ['o:gender','o:re_admission', 'o:age', 'o:Weight_kg']
obs_keep_cols = ['o:gender','o:age','o:Weight_kg',"o:PT","o:Platelets_count","o:INR","o:GCS",\
           "o:HR","o:Glucose","o:Creatinine","o:Hb","o:Temp_C","o:WBC_count","o:PTT"]

dem_cols = [i for i in train_data.columns if i in dem_keep_cols]
obs_cols = [i for i in train_data.columns if i in obs_keep_cols]
ac_cols  = [i for i in train_data.columns if i == 'o:heparin_1hourly']
rew_cols = [i for i in train_data.columns if i == 'PTT_reward']
#Assuming discrete actions and scalar rewards:

assert len(obs_cols) > 0, 'No observations present, or observation columns not prefixed with "o:"'
assert len(ac_cols) > 0, 'No actions present, or actions column not prefixed with "a:"'
assert len(rew_cols) > 0, 'No rewards present, or rewards column not prefixed with "r:"'
assert len(rew_cols) == 1, 'Multiple reward columns are present when a single reward column is expected'
# assert len(acuity_cols) == num_acuity_scores, 'Ensure that we have the right number of acuity scores'
ac_col = ac_cols[0]
rew_col = rew_cols[0]


In [ ]:
print("Converting Training Data")
print("--"*20)
train_data[ac_col] = train_data[ac_col]
all_actions = train_data[ac_col].unique()
all_actions.sort()
try:
    all_actions = all_actions.astype(np.float64)
except:
    raise ValueError('Actions are expected to be integers, but are not.')
# if not all(all_actions == np.arange(num_actions, dtype=np.int32)):
    # print(Font.red + 'Some actions are missing from data or all action space not properly defined.' + Font.end)
data_trajectory = {}
data_trajectory['dem_cols'] = dem_cols
data_trajectory['obs_cols'] = obs_cols
data_trajectory['ac_col']  = ac_col
data_trajectory['rew_col'] = rew_col
data_trajectory['num_actions'] = num_actions
data_trajectory['obs_dim'] = len(obs_cols)
data_trajectory['traj'] = {}
data_trajectory['pos_traj'] = []
data_trajectory['neg_traj'] = []

length = list()
for i in trajectories:
    # bar.update()
    traj_i = train_data[train_data['traj'] == i].sort_values(by='step')
    sequence = torch.Tensor(traj_i[dem_cols].values).to('cpu').shape[0]
    donse = torch.Tensor(np.array([False]*(sequence-1) + [True]))
    length.append(len(donse))
    data_trajectory['traj'][i] = {}
    data_trajectory['traj'][i]['dem'] = torch.Tensor(traj_i[dem_cols].values).to('cpu')
    data_trajectory['traj'][i]['obs'] = torch.Tensor(traj_i[obs_cols].values).to('cpu')
    data_trajectory['traj'][i]['actions'] = torch.Tensor(traj_i[ac_col].values.astype(np.float64)).to('cpu')
    data_trajectory['traj'][i]['rewards'] = torch.Tensor(traj_i[rew_col].values).to('cpu')
    data_trajectory['traj'][i]['done'] = torch.Tensor(np.array([False]*(sequence-1) + [True])).to('cpu')
    if sum(traj_i[rew_col].values) > 0:
        data_trajectory['pos_traj'].append(i)
    else:
        data_trajectory['neg_traj'].append(i)

In [ ]:
num_state = 14
train_buffer = Buffer(len(train_data), num_state)

In [ ]:
count=0
remove = 0
for i in data_trajectory['traj']:
    traj = data_trajectory['traj'][i] 
    dems = traj['dem']
    obs = traj['obs']
    actions = traj['actions']
    rewards = traj['rewards']
    dones = traj['done']
    for j in range(dems.shape[0]):
        try:
            dem = np.array(dems[j])
            ob = np.array(obs[j])
            a = np.array(actions[j])
            r = np.array(rewards[j])
            done = np.array(dones[j])
            next_state = np.array(obs[j+1])
            train_buffer.record((ob, a, r, next_state,done))
            count+=1
        except:
            remove+=1
            
print(len(train_data))
print(count+remove)

In [ ]:
s = train_buffer.state_buffer
a = train_buffer.action_buffer
r = train_buffer.reward_buffer
n_s = train_buffer.next_state_buffer
d = train_buffer.done_buffer

np.save("./final_result/1hourly/train_s",s)
np.save("./final_result/1hourly/train_a",a)
np.save("./final_result/1hourly/train_r",r)
np.save("./final_result/1hourly/train_ns",n_s)
np.save("./final_result/1hourly/train_d",d)

# validation buffer 저장

In [ ]:
dem_keep_cols = ['o:gender','o:re_admission', 'o:age', 'o:Weight_kg']
obs_keep_cols = ['o:gender','o:age','o:Weight_kg',"o:PT","o:Platelets_count","o:INR","o:GCS",\
           "o:HR","o:Glucose","o:Creatinine","o:Hb","o:Temp_C","o:WBC_count","o:PTT"]

dem_cols = [i for i in val_data.columns if i in dem_keep_cols]
obs_cols = [i for i in val_data.columns if i in obs_keep_cols]
ac_cols  = [i for i in val_data.columns if i == 'o:heparin_1hourly']
rew_cols = [i for i in val_data.columns if i == 'PTT_reward']
#Assuming discrete actions and scalar rewards:

assert len(obs_cols) > 0, 'No observations present, or observation columns not prefixed with "o:"'
assert len(ac_cols) > 0, 'No actions present, or actions column not prefixed with "a:"'
assert len(rew_cols) > 0, 'No rewards present, or rewards column not prefixed with "r:"'
assert len(rew_cols) == 1, 'Multiple reward columns are present when a single reward column is expected'
# assert len(acuity_cols) == num_acuity_scores, 'Ensure that we have the right number of acuity scores'
ac_col = ac_cols[0]
rew_col = rew_cols[0]

In [ ]:
print("Converting validation Data")
print("--"*20)
val_data[ac_col] = val_data[ac_col]
all_actions = val_data[ac_col].unique()
all_actions.sort()
try:
    all_actions = all_actions.astype(np.float64)
except:
    raise ValueError('Actions are expected to be integers, but are not.')
# if not all(all_actions == np.arange(num_actions, dtype=np.int32)):
    # print(Font.red + 'Some actions are missing from data or all action space not properly defined.' + Font.end)
data_trajectory = {}
data_trajectory['dem_cols'] = dem_cols
data_trajectory['obs_cols'] = obs_cols
data_trajectory['ac_col']  = ac_col
data_trajectory['rew_col'] = rew_col
data_trajectory['num_actions'] = num_actions
data_trajectory['obs_dim'] = len(obs_cols)
data_trajectory['traj'] = {}
data_trajectory['pos_traj'] = []
data_trajectory['neg_traj'] = []

length = list()
for i in val_trajectories:
    # bar.update()
    traj_i = val_data[val_data['traj'] == i].sort_values(by='step')
    sequence = torch.Tensor(traj_i[dem_cols].values).to('cpu').shape[0]
    donse = torch.Tensor(np.array([False]*(sequence-1) + [True]))
    length.append(len(donse))
    data_trajectory['traj'][i] = {}
    data_trajectory['traj'][i]['dem'] = torch.Tensor(traj_i[dem_cols].values).to('cpu')
    data_trajectory['traj'][i]['obs'] = torch.Tensor(traj_i[obs_cols].values).to('cpu')
    data_trajectory['traj'][i]['actions'] = torch.Tensor(traj_i[ac_col].values.astype(np.float64)).to('cpu')
    data_trajectory['traj'][i]['rewards'] = torch.Tensor(traj_i[rew_col].values).to('cpu')
    data_trajectory['traj'][i]['done'] = torch.Tensor(np.array([False]*(sequence-1) + [True])).to('cpu')
    if sum(traj_i[rew_col].values) > 0:
        data_trajectory['pos_traj'].append(i)
    else:
        data_trajectory['neg_traj'].append(i)

In [ ]:
num_state = 14
val_buffer = Buffer(len(val_data), num_state)

In [ ]:
count=0
remove = 0
for i in data_trajectory['traj']:
    traj = data_trajectory['traj'][i] 
    dems = traj['dem']
    obs = traj['obs']
    actions = traj['actions']
    rewards = traj['rewards']
    dones = traj['done']
    for j in range(dems.shape[0]):
        try:
            dem = np.array(dems[j])
            ob = np.array(obs[j])
            a = np.array(actions[j])
            r = np.array(rewards[j])
            done = np.array(dones[j])
            next_state = np.array(obs[j+1])
            val_buffer.record((ob, a, r, next_state,done))
            count+=1
        except:
            remove+=1
            
print(len(val_data))
print(count+remove)

In [ ]:
s = val_buffer.state_buffer
a = val_buffer.action_buffer
r = val_buffer.reward_buffer
n_s = val_buffer.next_state_buffer
d = val_buffer.done_buffer

np.save("./final_result/1hourly/val_s",s)
np.save("./final_result/1hourly/val_a",a)
np.save("./final_result/1hourly/val_r",r)
np.save("./final_result/1hourly/val_ns",n_s)
np.save("./final_result/1hourly/val_d",d)

# test buffer 저장

In [ ]:
dem_keep_cols = ['o:gender','o:re_admission', 'o:age', 'o:Weight_kg']
obs_keep_cols = ['o:gender','o:age','o:Weight_kg',"o:PT","o:Platelets_count","o:INR","o:GCS",\
           "o:HR","o:Glucose","o:Creatinine","o:Hb","o:Temp_C","o:WBC_count","o:PTT"]

dem_cols = [i for i in test_data.columns if i in dem_keep_cols]
obs_cols = [i for i in test_data.columns if i in obs_keep_cols]
ac_cols  = [i for i in test_data.columns if i == 'o:heparin_1hourly']
rew_cols = [i for i in test_data.columns if i == 'PTT_reward']
#Assuming discrete actions and scalar rewards:

assert len(obs_cols) > 0, 'No observations present, or observation columns not prefixed with "o:"'
assert len(ac_cols) > 0, 'No actions present, or actions column not prefixed with "a:"'
assert len(rew_cols) > 0, 'No rewards present, or rewards column not prefixed with "r:"'
assert len(rew_cols) == 1, 'Multiple reward columns are present when a single reward column is expected'
# assert len(acuity_cols) == num_acuity_scores, 'Ensure that we have the right number of acuity scores'
ac_col = ac_cols[0]
rew_col = rew_cols[0]

In [ ]:
print("Converting Testing Data")
print("--"*20)
test_data[ac_col] = test_data[ac_col]
all_actions = test_data[ac_col].unique()
all_actions.sort()
try:
    all_actions = all_actions.astype(np.float64)
except:
    raise ValueError('Actions are expected to be integers, but are not.')
# if not all(all_actions == np.arange(num_actions, dtype=np.int32)):
    # print(Font.red + 'Some actions are missing from data or all action space not properly defined.' + Font.end)
data_trajectory = {}
data_trajectory['dem_cols'] = dem_cols
data_trajectory['obs_cols'] = obs_cols
data_trajectory['ac_col']  = ac_col
data_trajectory['rew_col'] = rew_col
data_trajectory['num_actions'] = num_actions
data_trajectory['obs_dim'] = len(obs_cols)
data_trajectory['traj'] = {}
data_trajectory['pos_traj'] = []
data_trajectory['neg_traj'] = []

length = list()
for i in test_trajectories:
    # bar.update()
    traj_i = test_data[test_data['traj'] == i].sort_values(by='step')
    sequence = torch.Tensor(traj_i[dem_cols].values).to('cpu').shape[0]
    donse = torch.Tensor(np.array([False]*(sequence-1) + [True]))
    length.append(len(donse))
    data_trajectory['traj'][i] = {}
    data_trajectory['traj'][i]['dem'] = torch.Tensor(traj_i[dem_cols].values).to('cpu')
    data_trajectory['traj'][i]['obs'] = torch.Tensor(traj_i[obs_cols].values).to('cpu')
    data_trajectory['traj'][i]['actions'] = torch.Tensor(traj_i[ac_col].values.astype(np.float64)).to('cpu')
    data_trajectory['traj'][i]['rewards'] = torch.Tensor(traj_i[rew_col].values).to('cpu')
    data_trajectory['traj'][i]['done'] = torch.Tensor(np.array([False]*(sequence-1) + [True])).to('cpu')
    if sum(traj_i[rew_col].values) > 0:
        data_trajectory['pos_traj'].append(i)
    else:
        data_trajectory['neg_traj'].append(i)

In [ ]:
num_state = 14
test_buffer = Buffer(len(test_data), num_state)

In [ ]:
count=0
remove = 0
for i in data_trajectory['traj']:
    traj = data_trajectory['traj'][i] 
    dems = traj['dem']
    obs = traj['obs']
    actions = traj['actions']
    rewards = traj['rewards']
    dones = traj['done']
    for j in range(dems.shape[0]):
        try:
            dem = np.array(dems[j])
            ob = np.array(obs[j])
            a = np.array(actions[j])
            r = np.array(rewards[j])
            done = np.array(dones[j])
            next_state = np.array(obs[j+1])
            test_buffer.record((ob, a, r, next_state,done))
            count+=1
        except:
            remove+=1
            
print(len(test_data))
print(count+remove)

In [ ]:
s = test_buffer.state_buffer
a = test_buffer.action_buffer
r = test_buffer.reward_buffer
n_s = test_buffer.next_state_buffer
d = test_buffer.done_buffer

np.save("./final_result/1hourly/test_s",s)
np.save("./final_result/1hourly/test_a",a)
np.save("./final_result/1hourly/test_r",r)
np.save("./final_result/1hourly/test_ns",n_s)
np.save("./final_result/1hourly/test_d",d)